# Laboratori 11: Bredhjet e rastit, difuzioni dhe zinxhirët Markov

**Lënda:** Simulim dhe Metoda Numerike  
**Lidhja me leksionin:** Kapitulli 11  
**Kohëzgjatja e rekomanduar:** 2 orë  
**Botimi:** Versioni i studentit

> **Pyetja udhëheqëse:** Si lind difuzioni nga bredhja mikroskopike dhe si diagnostikohet varësia në një zinxhir Markov?

Notebook-u ndahet në dy gjysma të barasvlershme. Gjysma e parë përdor Python; gjysma e dytë përdor MATLAB/Octave. Studenti mund të zgjedhë njërën gjuhë, por duhet të raportojë të njëjtat kontrolle numerike dhe interpretime fizike.


## Rezultatet e të nxënit

Në përfundim, studenti duhet të jetë në gjendje:

- të verifikojë $\langle x^2\rangle\propto t$;
- të vlerësojë kohën e parë të kalimit;
- të llogarisë shpërndarjen stacionare;
- të vlerësojë autokorrelacionin dhe madhësinë efektive të mostrës;


## Përgatitja teorike


Për bredhjen simetrike me hapa $\pm\ell$, $\mathbb E[X_N]=0$ dhe $\mathbb E[X_N^2]=N\ell^2$. Në kufirin $\ell,\Delta t\to0$ me $D=\ell^2/(2\Delta t)$ të fiksuar, merret ekuacioni i difuzionit $p_t=Dp_{xx}$.

Një matricë kalimi $P$ ka shpërndarje stacionare $\boldsymbol\pi$ kur $\boldsymbol\pi^TP=\boldsymbol\pi^T$. Balanca e detajuar është $\pi_iP_{ij}=\pi_jP_{ji}$. Për mostra të korreluara,
$$
N_{\mathrm{eff}}\approx\frac{N}{2\tau_{\mathrm{int}}},\qquad \tau_{\mathrm{int}}=\frac12+\sum_{k\ge1}\rho(k).
$$


## Algoritmi dhe kriteret e verifikimit

Simulojmë shumë bredhje, masim zhvendosjen katrore mesatare dhe kohën e daljes, pastaj ndërtojmë një zinxhir me tri gjendje dhe krahasojmë shpërndarjen empirike me atë stacionare.

Në çdo eksperiment raportohen parametrat, madhësia e rrjetës ose e mostrës, kriteri i ndalimit, një mbetje ose ligj ruajtjeje dhe një vlerësim i gabimit.


# Pjesa I — Python

Ekzekutoni qelizat sipas radhës. Ndryshoni vetëm një faktor në çdo eksperiment numerik dhe ruani figurat që mbështesin përfundimin.


### Eksperimenti Python 1


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (8, 4.6),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})
rng = np.random.default_rng(2026)


### Eksperimenti Python 2


In [ ]:
M=5000; Ns=np.unique(np.logspace(1,3.5,25).astype(int)); msd=[]
for N in Ns:
    pos=rng.choice([-1,1],(M,N)).sum(axis=1); msd.append(np.mean(pos**2))
print('Eksponenti:',np.polyfit(np.log(Ns),np.log(msd),1)[0])
plt.loglog(Ns,msd,'o',label='simulimi'); plt.loglog(Ns,Ns,label='$N$'); plt.legend(); plt.show()


### Eksperimenti Python 3


In [ ]:
def koha_daljes(L=20,M=10000):
    x=np.zeros(M,dtype=int); active=np.ones(M,bool); tau=np.zeros(M,int); n=0
    while active.any():
        n+=1; x[active]+=rng.choice([-1,1],active.sum()); hit=active&(np.abs(x)>=L); tau[hit]=n; active[hit]=False
    return tau
tau=koha_daljes(); print('Koha mesatare:',tau.mean(),'teoria afërsisht:',20**2)


### Eksperimenti Python 4


In [ ]:
P=np.array([[.8,.15,.05],[.2,.6,.2],[.1,.2,.7]])
vals,vecs=np.linalg.eig(P.T); pi=np.real(vecs[:,np.argmin(abs(vals-1))]); pi/=pi.sum()
N=100_000; s=np.empty(N,dtype=int); s[0]=0
for n in range(N-1): s[n+1]=rng.choice(3,p=P[s[n]])
freq=np.bincount(s,minlength=3)/N; print('empirike:',freq,'stacionare:',pi)


### Eksperimenti Python 5


In [ ]:
def acf(x,kmax=200):
    # TODO: plotësoni zbatimin duke u mbështetur te algoritmi i mësipërm.
    raise NotImplementedError('Plotësoni këtë pjesë')
rho=acf((s==0).astype(float)); positive=rho[1:][rho[1:]>0]; tau_int=.5+positive.sum(); print('tau_int=',tau_int,'N_eff=',N/(2*tau_int))


## Interpretimi i rezultateve të Python-it

Përgjigjuni me fjali të plota:

1. Cili rezultat verifikon drejtpërdrejt algoritmin?
2. Në cilin regjim rezultati është i pavarur nga hapi, madhësia e mostrës ose gjendja fillestare?
3. Cili është burimi dominues i gabimit ose i pacaktueshmërisë?
4. A mbështetet përfundimi fizik nga figura dhe nga një madhësi numerike?


# Pjesa II — MATLAB/Octave

Qelizat vijuese janë të tipit `raw` që notebook-u të mbetet i hapshëm me kernelin Python. Kopjojini në MATLAB ose Octave, ose ndryshojini në qeliza kodi kur përdorni një kernel MATLAB/Octave.


### Eksperimenti MATLAB/Octave 1


## Krahasimi ndërmjet gjuhëve

Kontrolloni që Python dhe MATLAB/Octave japin rezultate në përputhje me tolerancën e arsyeshme. Ndryshimet në paraqitjen grafike nuk janë dallime shkencore; dallim shkencor është ndryshimi i vlerës, rendit të konvergjencës, mbetjes, energjisë ose intervalit të besimit.


## Produkti që dorëzohet

Dorëzimi duhet të përmbajë:

- notebook-un e ekzekutuar ose skriptin MATLAB/Octave;
- figurat me boshte, njësi dhe legjendë;
- një tabelë të shkurtër me parametrat dhe rezultatet;
- një kontroll verifikimi;
- një paragraf interpretimi fizik;
- një paragraf mbi kufizimet dhe burimet e gabimit;
- funksionet e shkruara nga studenti, të organizuara në mënyrë që të mund të përfshihen më vonë në paketën e tij/saj.


## Kontroll i shpejtë

- A përdor kodi farë të dokumentuar kur ka rastësi?
- A dallohen gabimi numerik dhe pacaktueshmëria e hyrjeve?
- A janë ruajtur njësitë fizike?
- A është kontrolluar rezultati me një kufi, zgjidhje reference, mbetje ose ligj ruajtjeje?
- A mund të riprodhohet rezultati nga një student tjetër?
